In [ ]:
# ===========================================================================
# UA-SPEECH MODEL TRAINING - interactive driver
#
# All training logic lives in src/training/; this notebook only calls it and
# stores results, matching notebooks/01/02's convention - functions and
# architecture belong in src/, only the act of running training and storing
# models happens here.
#
#   src/training/models.py     model factory: acoustic / deep_frozen / deep_lora / fusion
#   src/training/runner.py     TrainingConfig, run_training() - the fold loop
#   src/training/baseline.py   Phase 2: frozen wav2vec + linear SVM baseline
#   src/training/engine.py     one epoch: AMP, gradient clipping, optimizer
#   src/training/reporting.py  predictions / metrics / confusion-matrix / ROC / embeddings I/O
#
# Every run below is a TrainingConfig + run_training() pair - the "experiment
# manager" IS TrainingConfig (src/training/runner.py): every
# checkpoint/metric/prediction/confusion-matrix/ROC/embedding already lands
# under outputs/<kind>/<run_name>/, keyed by run_name, so adding a future
# ablation means appending a TrainingConfig to a family list below, not
# writing new plumbing.
#
# See ROADMAP.md for the phase plan this notebook implements (Phase 1 sanity
# check, Phase 2 baseline reproduction, Phase 3 ablation).
# ===========================================================================

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config
from src.console import print_header, print_kv
from src.training.data import load_manifest

config.ensure_directories()

df_m6 = load_manifest()

print_header("UA-Speech Training Notebook")
print_kv("Manifest", config.MANIFEST_PATH)
print_kv("Utterances", len(df_m6))
print_kv("Speakers", df_m6["Speaker_ID"].nunique())

In [ ]:
# EXPERIMENT MANAGER - the three pathway families every ablation variant
# (src.training.models.MODEL_NAMES) belongs to. Grouping by family, rather
# than looping over all six variants in one cell, is what lets MFCC-only,
# Wav2Vec2-only, and Fusion training be run, resumed, or extended
# independently in the stages below - a future ablation variant is added by
# appending its model name to the right family list here, nothing else
# changes.
MFCC_FAMILY = ["acoustic"]
WAV2VEC_FAMILY = ["deep_frozen", "deep_lora"]
FUSION_FAMILY = ["fusion", "attention_fusion", "attention_fusion_praat"]

print_header("Experiment Manager - Pathway Families")
print_kv("MFCC-only", ", ".join(MFCC_FAMILY))
print_kv("Wav2Vec2-only", ", ".join(WAV2VEC_FAMILY))
print_kv("Fusion", ", ".join(FUSION_FAMILY))

In [ ]:
# STAGE 1 - Pipeline sanity check ("smoke test"). Trains the cheapest model
# (MFCC-only) for one fold, one epoch, on a tiny slice of data. This is NOT a
# real result - it exists to confirm the whole chain (model init, optimizer,
# scheduler, AMP, gradient clipping, early stopping, checkpointing,
# TensorBoard logging, prediction/metric/confusion-matrix/ROC/embedding
# writers) actually runs end to end before spending GPU time on a real run.
from src.training.runner import TrainingConfig, run_training

smoke_cfg = TrainingConfig(
    task="detection", model="acoustic",
    epochs=1, max_folds=1, limit_samples=24,
    run_name="_smoke_test",
)
run_training(df_m6, smoke_cfg)

In [ ]:
# STAGE 2 - Phase 2, step 1: reproduce the ICASSP base paper's feature
# extractor. Frozen wav2vec 2.0 (no LoRA, no fine-tuning) -> one 768-dim
# embedding per utterance PER HIDDEN-STATE LAYER (13 layers: the CNN
# feature-extractor output + 12 transformer layers). The base paper finds
# different layers win for different tasks (layer 1 for detection, layer 13
# for severity), so all 13 are extracted here rather than just the final
# layer - Stage 3 sweeps them to find which wins on this reproduction.
# Identical across every LOSO fold, so extracted once and cached to
# outputs/embeddings/.
from src.training.baseline import extract_frozen_embeddings_all_layers

frozen_embeddings_all_layers = extract_frozen_embeddings_all_layers(df_m6, batch_size=16)
print(f"Frozen embeddings (all layers): {frozen_embeddings_all_layers.shape}")

In [ ]:
# STAGE 3 - Phase 2, step 2: frozen wav2vec 2.0 -> linear SVM, swept across
# all 13 layers and evaluated on the full 28-fold LOSO detection protocol -
# exactly the base paper's pipeline and per-layer comparison. The paper's
# own reported result is layer 1 at 93.95% accuracy; the best layer found
# here is the number every other model in this project has to beat to be a
# genuine improvement, not an assumed one. If the best layer or accuracy
# lands far from the paper's, that's worth investigating (preprocessing,
# VAD, clip length) before trusting the ablation table in Stage 8.
from src.training.baseline import sweep_svm_baseline_layers

detection_layer_sweep = sweep_svm_baseline_layers(
    df_m6, task="detection", all_layer_embeddings=frozen_embeddings_all_layers, max_folds=None)

best_layer = int(detection_layer_sweep.iloc[0]["layer"])
baseline_pooled = detection_layer_sweep.iloc[0].to_dict()
baseline_pooled.pop("layer")

print_header("Baseline (Frozen wav2vec 2.0 + Linear SVM) - Detection")
print_kv("Best layer", f"{best_layer} (paper reports layer 1 at 93.95% accuracy)")
print_kv("Accuracy", f"{baseline_pooled['accuracy']:.4f}")
print_kv("F1", f"{baseline_pooled['f1']:.4f}")
print_kv("Recall (sensitivity)", f"{baseline_pooled['recall']:.4f}")
print_kv("Precision", f"{baseline_pooled['precision']:.4f}")
print_kv("Specificity", f"{baseline_pooled['specificity']:.4f}")
print_kv("AUROC", f"{baseline_pooled['auroc']:.4f}")

In [ ]:
# STAGE 4 - Phase 2, step 3: the same frozen wav2vec 2.0 + linear SVM layer
# sweep, on the severity task's 81-fold balanced leave-one-per-class-out
# protocol (config.DROPPED_FOR_BALANCE, corrected to match the base paper's
# stated exclusion criterion - see src/config.py). The paper's own reported
# result is layer 13 (final) at 44.56% accuracy (4-class) - a low absolute
# number, expected for a 4-way severity task, but the target to compare
# against before trusting the severity ablation in Stage 13.
from src.training.baseline import sweep_svm_baseline_layers

severity_layer_sweep = sweep_svm_baseline_layers(
    df_m6, task="severity", all_layer_embeddings=frozen_embeddings_all_layers, max_folds=None)

severity_best_layer = int(severity_layer_sweep.iloc[0]["layer"])
severity_baseline_pooled = severity_layer_sweep.iloc[0].to_dict()
severity_baseline_pooled.pop("layer")

print_header("Baseline (Frozen wav2vec 2.0 + Linear SVM) - Severity")
print_kv("Best layer", f"{severity_best_layer} (paper reports layer 13/final at 44.56% accuracy)")
print_kv("Accuracy", f"{severity_baseline_pooled['accuracy']:.4f}")
print_kv("F1", f"{severity_baseline_pooled['f1']:.4f}")

In [ ]:
# STAGE 5 - MFCC-only screening (detection). Cheap 8-fold speaker-grouped
# screening run (not full 28-fold LOSO) of the Acoustic Pathway alone
# (Model A) - the cheapest of the six ablation variants, and the one Stage
# 1's smoke test already exercised end to end.
# screening_pooled is seeded with the Stage 3 SVM baseline and grows across
# Stages 5-7 as each pathway family screens.
#
# NOTE ON SCALE: full 28-fold LOSO x all 6 ablation variants x (detection +
# severity) does not fit a 10-hour total-compute budget - a single
# wav2vec-fine-tuning variant's full LOSO pass alone can run tens of
# GPU-hours. So Stages 5-7 rank all six variants cheaply on two axes at
# once: CV_PROTOCOL="screening" (src.splits.iter_screening_folds:
# SCREENING_FOLDS speaker-grouped folds instead of 28 single-speaker folds)
# AND a shorter SCREENING_EPOCHS/SCREENING_PATIENCE budget than the full
# run's DEFAULT_EPOCHS=20/DEFAULT_PATIENCE=5 - screening only needs to rank
# variants relative to each other, not train any one of them to full
# convergence, and only Stage 8b spends full-length full-LOSO GPU time - on
# the top TOP_K_FOR_DETECTION winners, not all six. Every training cell
# below is still session-bounded (SESSION_BUDGET_HOURS) and resumes on
# re-run via run_training()'s on-disk fold cache (src/training/runner.py).
from src.training.runner import TrainingConfig, run_training
from src.console import print_note
import time

# 6, not 2.5: on a local GPU (no free-Colab 2-hour session cap), the
# deadline is just a safety net against a crash/thermal-throttle/needing
# the machine back mid-run, not a hard external limit - lower it for a
# supervised first session to see real fold timing, then raise it (or set
# very high) once you trust the resume-on-rerun path to pick up cleanly
# after an interruption.
SESSION_BUDGET_HOURS = 6
SCREENING_FOLDS = 8         # speaker-grouped folds used to rank all 6 variants before full LOSO
SCREENING_EPOCHS = 10       # half of DEFAULT_EPOCHS - ranking needs relative signal, not convergence
SCREENING_PATIENCE = 3      # tighter than DEFAULT_PATIENCE=5 - stop unpromising variants sooner
deadline = time.monotonic() + SESSION_BUDGET_HOURS * 3600

screening_pooled = {"baseline_svm": baseline_pooled}

for model_name in MFCC_FAMILY:
    if time.monotonic() >= deadline:
        print_note(f"Session budget used up before starting '{model_name}' - "
                   "re-run this cell later to continue.")
        break
    cfg = TrainingConfig(
        task="detection", model=model_name,
        run_name=f"detection_screen_{model_name}",
        cv_protocol="screening", screening_folds=SCREENING_FOLDS,
        epochs=SCREENING_EPOCHS, patience=SCREENING_PATIENCE,
    )
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    if pooled:
        screening_pooled[model_name] = pooled

In [ ]:
# STAGE 6 - Wav2Vec2-only screening (detection). Frozen and LoRA-adapted
# wav2vec 2.0 + MLP head (Models B and C), on the same cheap screening
# protocol and shortened epoch budget as Stage 5. Continues screening_pooled
# from Stage 5; same session-bounded, resume-on-rerun pattern.
from src.training.runner import TrainingConfig, run_training
from src.console import print_note
import time

deadline = time.monotonic() + SESSION_BUDGET_HOURS * 3600

for model_name in WAV2VEC_FAMILY:
    if time.monotonic() >= deadline:
        print_note(f"Session budget used up before starting '{model_name}' - "
                   "re-run this cell later to continue.")
        break
    cfg = TrainingConfig(
        task="detection", model=model_name,
        run_name=f"detection_screen_{model_name}",
        cv_protocol="screening", screening_folds=SCREENING_FOLDS,
        epochs=SCREENING_EPOCHS, patience=SCREENING_PATIENCE,
    )
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    if pooled:
        screening_pooled[model_name] = pooled

In [ ]:
# STAGE 7 - Fusion model screening (detection). Concatenated Fusion (Model
# D) and the two Phase 6 attention-fusion variants (Models E, F), on the
# same cheap screening protocol and shortened epoch budget as Stages 5-6.
# Continues screening_pooled from Stages 5-6; same session-bounded,
# resume-on-rerun pattern.
from src.training.runner import TrainingConfig, run_training
from src.console import print_note
import time

deadline = time.monotonic() + SESSION_BUDGET_HOURS * 3600

for model_name in FUSION_FAMILY:
    if time.monotonic() >= deadline:
        print_note(f"Session budget used up before starting '{model_name}' - "
                   "re-run this cell later to continue.")
        break
    cfg = TrainingConfig(
        task="detection", model=model_name,
        run_name=f"detection_screen_{model_name}",
        cv_protocol="screening", screening_folds=SCREENING_FOLDS,
        epochs=SCREENING_EPOCHS, patience=SCREENING_PATIENCE,
    )
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    if pooled:
        screening_pooled[model_name] = pooled

In [ ]:
# STAGE 8 - Screening comparison table: baseline SVM vs. every screened
# variant from Stages 5-7, pooled metrics side by side. This ranks the six
# ablation variants cheaply; it is NOT the reportable LOSO result (Stage 8b
# runs that for the winners only). Saved to
# outputs/metrics/phase2_screening_comparison.csv.
from src.model_analysis import plot_ablation_comparison
from src.results import style_comparison_table

screening_df = pd.DataFrame(screening_pooled).T
screening_df.index.name = "model"

screening_path = config.METRICS_DIR / "phase2_screening_comparison.csv"
screening_df.to_csv(screening_path)

print_header("Phase 2/3 Screening Comparison - Detection")
print_kv("Saved to", screening_path)

loss_cols = [c for c in screening_df.columns if "loss" in c]
score_cols = [c for c in screening_df.columns if c not in loss_cols]
display(style_comparison_table(screening_df[score_cols]))
if loss_cols:
    display(style_comparison_table(screening_df[loss_cols], higher_is_better=False))

plot_ablation_comparison(screening_df, title="Ablation Screening - Detection", show=True)

In [ ]:
# STAGE 8b - Full 28-fold LOSO training (detection), winners only. This is
# where the compute budget actually gets spent deliberately: instead of
# full LOSO x all 6 variants, only the top TOP_K_FOR_DETECTION variants from
# Stage 8's cheap screening get the real base-paper protocol. That is what
# makes comparison_pooled below the number worth reporting/citing - the
# screening_pooled numbers above rank variants but are not full LOSO.
# Same session-bounded, resume-on-rerun pattern as Stages 5-7.
from src.training.runner import TrainingConfig, run_training
from src.training.models import MODEL_NAMES
from src.console import print_note
import time

TOP_K_FOR_DETECTION = 2

ranked_by_screening = sorted(
    (m for m in MODEL_NAMES if m in screening_pooled),
    key=lambda m: screening_pooled[m]["f1"], reverse=True,
)
detection_loso_candidates = ranked_by_screening[:TOP_K_FOR_DETECTION]
print_note(f"Running full 28-fold LOSO only for top {TOP_K_FOR_DETECTION} "
          f"screened variants by F1: {detection_loso_candidates}")

SESSION_BUDGET_HOURS = 6  # safety net, not a hard cap - see Stage 5's comment
deadline = time.monotonic() + SESSION_BUDGET_HOURS * 3600

comparison_pooled = {"baseline_svm": baseline_pooled}

for model_name in detection_loso_candidates:
    if time.monotonic() >= deadline:
        print_note(f"Session budget used up before starting '{model_name}' - "
                   "re-run this cell later to continue.")
        break
    cfg = TrainingConfig(
        task="detection", model=model_name,
        run_name=f"detection_{model_name}",
    )
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    if pooled:
        comparison_pooled[model_name] = pooled

In [ ]:
# STAGE 8c - Phase 2/3 comparison table: baseline SVM vs. Stage 8b's full-LOSO
# winners, pooled metrics side by side. This is the reportable, base-paper-
# comparable detection result (unlike Stage 8's screening table). Saved to
# outputs/metrics/phase2_comparison.csv for the paper/report.
from src.model_analysis import plot_ablation_comparison
from src.results import style_comparison_table

comparison_df = pd.DataFrame(comparison_pooled).T
comparison_df.index.name = "model"

comparison_path = config.METRICS_DIR / "phase2_comparison.csv"
comparison_df.to_csv(comparison_path)

print_header("Phase 2/3 Comparison - Detection (full LOSO)")
print_kv("Saved to", comparison_path)

loss_cols = [c for c in comparison_df.columns if "loss" in c]
score_cols = [c for c in comparison_df.columns if c not in loss_cols]
display(style_comparison_table(comparison_df[score_cols]))
if loss_cols:
    display(style_comparison_table(comparison_df[loss_cols], higher_is_better=False))

plot_ablation_comparison(comparison_df, title="Ablation Comparison - Detection (Full LOSO)", show=True)

In [ ]:
# STAGE 9 - Pick which variant(s) go on to the severity task. Stages 10-12
# below run the severity protocol (leave-one-speaker-per-class-out,
# subsampled to SEVERITY_FOLD_SAMPLE of the base paper's 81 combinations -
# see Stage 10), the single largest remaining GPU-time item in this
# notebook. Only the architecture that actually won Stage 8c's full-LOSO
# detection comparison matters for the severity story, so rank those
# results by F1 and carry forward just the top TOP_K_FOR_SEVERITY (picked
# from detection_loso_candidates, not all six variants - the others never
# got a full-LOSO score to rank by).
TOP_K_FOR_SEVERITY = 1

ranked_variants = sorted(
    (m for m in detection_loso_candidates if m in comparison_pooled),
    key=lambda m: comparison_pooled[m]["f1"], reverse=True,
)
severity_model_names = ranked_variants[:TOP_K_FOR_SEVERITY]

print_note(f"Running severity only for top {TOP_K_FOR_SEVERITY} "
          f"full-LOSO detection variant(s) by F1: {severity_model_names}")

In [ ]:
# STAGE 10 - Severity training - MFCC-only. Runs src.training.runner over
# MFCC_FAMILY, restricted to whichever of those variants made Stage 9's
# top-TOP_K_FOR_SEVERITY cut; if none did, the loop below is simply a no-op.
# Balanced leave-one-speaker-per-class-out protocol (src/splits.py,
# config.DROPPED_FOR_BALANCE), subsampled to SEVERITY_FOLD_SAMPLE of the
# base paper's 81 combinations (src.splits.sample_severity_folds) - 81 folds
# is a bigger job than detection's 28-fold LOSO even before Stage 9 already
# cut this down to one variant, and it's still the largest remaining
# GPU-time item in a 10-hour total-compute budget. Plus the Stage 4 severity
# SVM baseline for a like-for-like comparison table (mirrors Stages 5-8's
# detection pattern). Same session-bounded, resume-on-rerun pattern as
# Stages 5-7.
#
# Lower priority than Stages 5-8c's detection run (the paper's primary
# comparison) - run Stages 10-12 once detection is done or far enough along.
from src.training.runner import TrainingConfig, run_training
from src.console import print_note
import time

SESSION_BUDGET_HOURS = 6    # safety net, not a hard cap - see Stage 5's comment
SEVERITY_FOLD_SAMPLE = 20   # of the base paper's 81 leave-one-per-class-out combinations
deadline = time.monotonic() + SESSION_BUDGET_HOURS * 3600

severity_pooled = {"baseline_svm": severity_baseline_pooled}

for model_name in (m for m in MFCC_FAMILY if m in severity_model_names):
    if time.monotonic() >= deadline:
        print_note(f"Session budget used up before starting '{model_name}' - "
                   "re-run this cell later to continue.")
        break
    cfg = TrainingConfig(
        task="severity", model=model_name,
        run_name=f"severity_{model_name}",
        severity_fold_sample=SEVERITY_FOLD_SAMPLE,
    )
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    if pooled:
        severity_pooled[model_name] = pooled

In [ ]:
# STAGE 11 - Severity training - Wav2Vec2-only. Continues severity_pooled
# from Stage 10, restricted to whichever of WAV2VEC_FAMILY made the Stage 9
# cut. Same SEVERITY_FOLD_SAMPLE subsample as Stage 10.
from src.training.runner import TrainingConfig, run_training
from src.console import print_note
import time

deadline = time.monotonic() + SESSION_BUDGET_HOURS * 3600

for model_name in (m for m in WAV2VEC_FAMILY if m in severity_model_names):
    if time.monotonic() >= deadline:
        print_note(f"Session budget used up before starting '{model_name}' - "
                   "re-run this cell later to continue.")
        break
    cfg = TrainingConfig(
        task="severity", model=model_name,
        run_name=f"severity_{model_name}",
        severity_fold_sample=SEVERITY_FOLD_SAMPLE,
    )
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    if pooled:
        severity_pooled[model_name] = pooled

In [ ]:
# STAGE 12 - Severity training - Fusion. Continues severity_pooled from
# Stages 10-11, restricted to whichever of FUSION_FAMILY made the Stage 9
# cut. Same SEVERITY_FOLD_SAMPLE subsample as Stage 10.
from src.training.runner import TrainingConfig, run_training
from src.console import print_note
import time

deadline = time.monotonic() + SESSION_BUDGET_HOURS * 3600

for model_name in (m for m in FUSION_FAMILY if m in severity_model_names):
    if time.monotonic() >= deadline:
        print_note(f"Session budget used up before starting '{model_name}' - "
                   "re-run this cell later to continue.")
        break
    cfg = TrainingConfig(
        task="severity", model=model_name,
        run_name=f"severity_{model_name}",
        severity_fold_sample=SEVERITY_FOLD_SAMPLE,
    )
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    if pooled:
        severity_pooled[model_name] = pooled

In [ ]:
# STAGE 13 - Phase 3 severity comparison table: baseline SVM vs. Stages
# 10-12's severity-trained variant(s) (subsampled folds - see Stage 10),
# pooled metrics side by side. Saved to
# outputs/metrics/phase3_severity_comparison.csv for the paper/report. Same
# colour-graded display as Stage 8c.
from src.results import style_comparison_table

severity_df = pd.DataFrame(severity_pooled).T
severity_df.index.name = "model"

severity_path = config.METRICS_DIR / "phase3_severity_comparison.csv"
severity_df.to_csv(severity_path)

print_header("Phase 3 Comparison - Severity")
print_kv("Saved to", severity_path)

loss_cols = [c for c in severity_df.columns if "loss" in c]
score_cols = [c for c in severity_df.columns if c not in loss_cols]
display(style_comparison_table(severity_df[score_cols]))
if loss_cols:
    display(style_comparison_table(severity_df[loss_cols], higher_is_better=False))